
# K-Fold CV and LOOCV for Regression

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Cross-validation estimates prediction error when one train/test split is too dependent on a lucky or unlucky split.

**Highest-probability exam pattern:** Use k-fold CV or LOOCV to choose polynomial degree or another regression tuning parameter.



## Assumptions, intuition, and theory

K-fold CV fits k models and validates each fold once. LOOCV fits n models and validates one observation at a time. LOOCV uses more training data per fit but can be computationally expensive.



    ## Python method notes and documentation

    `KFold` defines folds, `LeaveOneOut` defines one-observation validation folds, and `cross_val_score` returns one score per fold. Negative MSE is used because sklearn scoring conventions make larger scores better.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [LeaveOneOut](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for CV summaries.
import matplotlib.pyplot as plt  # Import plotting tools for CV curves.
from sklearn.linear_model import LinearRegression  # Import ordinary least-squares regression.
from sklearn.metrics import mean_squared_error  # Import MSE for manual checking if needed.
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score  # Import k-fold, LOOCV, and scoring helper.
from sklearn.pipeline import make_pipeline  # Import pipeline construction.
from sklearn.preprocessing import PolynomialFeatures  # Import polynomial feature expansion.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a reusable nonlinear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw predictor values and sort them for cleaner plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true nonlinear mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian noise to create observed responses.
    X = x.reshape(-1, 1)  # Convert the predictor to a two-dimensional sklearn design matrix.
    return X, y, signal  # Return predictors, observed response, and noiseless truth.

def make_polynomial_regression_data(n=140, noise=2.0, random_state=123):  # Define a curved polynomial regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = rng.uniform(-2.5, 2.5, size=n)  # Draw one numeric predictor over a moderate range.
    signal = 1.0 - 2.0 * x + 0.8 * x**2 + 0.7 * x**3  # Define the true polynomial mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian errors.
    X = x.reshape(-1, 1)  # Reshape the predictor for sklearn estimators.
    return X, y, signal  # Return the design matrix, response, and true mean.

def train_test_mse(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test regression evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split observations into training and testing parts.
        X,  # Pass the predictor matrix to the splitter.
        y,  # Pass the numeric response vector to the splitter.
        test_size=test_size,  # Hold out this fraction for honest testing.
        random_state=random_state,  # Fix the random split for reproducibility.
    )  # Finish the train/test split call.
    fitted_model = clone(model)  # Clone the estimator so repeated calls do not reuse fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the model only on the training data.
    train_pred = fitted_model.predict(X_train)  # Predict the training responses to assess in-sample error.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out responses to assess future error.
    train_mse = mean_squared_error(y_train, train_pred)  # Compute training mean squared error.
    test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
    return fitted_model, train_mse, test_mse  # Return the fitted model and both error estimates.


In [ ]:
X, y, true_signal = make_sine_regression_data(n=75, noise=0.35, random_state=RANDOM_SEED)  # Simulate a small nonlinear regression data set.
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)  # Define shuffled 5-fold cross-validation.
loocv = LeaveOneOut()  # Define leave-one-out cross-validation.
rows = []  # Create a list for CV results.
for degree in range(1, 8):  # Try candidate polynomial degrees.
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())  # Build the polynomial regression pipeline.
    kfold_mse = -cross_val_score(model, X, y, cv=kfold, scoring='neg_mean_squared_error')  # Convert sklearn negative MSE scores to positive MSE values.
    loo_mse = -cross_val_score(model, X, y, cv=loocv, scoring='neg_mean_squared_error')  # Compute LOOCV pointwise MSE values.
    rows.append({'degree': degree, 'kfold_mse': np.mean(kfold_mse), 'loocv_mse': np.mean(loo_mse)})  # Store average CV errors.
cv_results = pd.DataFrame(rows)  # Convert CV results to a table.
best_degree = int(cv_results.loc[cv_results['kfold_mse'].idxmin(), 'degree'])  # Choose the degree with smallest 5-fold CV MSE.
display(cv_results)  # Display the CV comparison table.
print(f'Best degree by 5-fold CV: {best_degree}')  # Print the selected degree.
plt.figure(figsize=(6, 4))  # Create a CV error-curve figure.
plt.plot(cv_results['degree'], cv_results['kfold_mse'], marker='o', label='5-fold CV MSE')  # Plot k-fold CV error.
plt.plot(cv_results['degree'], cv_results['loocv_mse'], marker='o', label='LOOCV MSE')  # Plot LOOCV error.
plt.xlabel('polynomial degree')  # Label the tuning axis.
plt.ylabel('estimated prediction MSE')  # Label the error axis.
plt.title('K-fold CV and LOOCV choose model complexity')  # Title the plot.
plt.legend()  # Show the CV method labels.
plt.show()  # Render the plot.



## Exam-style problem prompt

A problem asks for LOOCV or k-fold CV to choose a regression model. Compute CV MSE for each candidate and choose the smallest estimated prediction error.



    ## Adaptable code pattern

    ```python
    # Step 1: Define candidate models or tuning values.
# Step 2: Create KFold or LeaveOneOut splitter.
# Step 3: Use cross_val_score with scoring="neg_mean_squared_error".
# Step 4: Multiply scores by -1 to recover positive MSE.
# Step 5: Average across folds.
# Step 6: Choose the tuning value with smallest average CV MSE.
    ```



## What to remember for the exam

In sklearn, negative MSE is not a mistake. It is a scoring convention. For interpretation, multiply by -1 and minimize MSE.
